# Recherche sectorielle EU / US

Ce notebook permet de configurer le modèle, vérifier les variables actives, lancer le calcul puis exécuter les visualisations et backtests.


In [ ]:
# Paramètres du modèle

UNIVERSE = "EU"          # "EU" ou "US"

# None = méthodologie par défaut.
# Exemple :
# MODEL_VARIABLES = {
#     "Value": ["price_fcf", "ev_ebitda"],
#     "Momentum": ["momentum_6m_1m", "earnings_revision_ratio"],
# }
MODEL_VARIABLES = None

# Pilier étudié dans les cellules de détail.
PILLAR = "Value"

# None = toutes les variables actives du pilier.
BACKTEST_VARIABLES = None

# None = tous les piliers dans la heatmap.
PILLARS_PLOT = None

# None = dernière date disponible.
DATE = None

# Paramètres du backtest.
TOP_N = 3
START = None             # Exemple : "2015-01-31"
END = None
INCLUDE_PILLAR = True

# Activation des graphiques.
PLOT_PILLARS = True
PLOT_VARIABLES = True
PLOT_GLOBAL_RANK = True
RUN_BACKTEST = True
BACKTEST_PLOT = True

# Choix possibles : "top", "worst", "long_short", "benchmark"
PLOT_SERIES = ["top", "benchmark"]

# Sauvegarde optionnelle du backtest.
SAVE_BACKTEST = False
BACKTEST_OUTPUT_DIR = "output/backtests"

# None = fichiers configurés localement.
EXCEL_FILE = None
MACRO_FILE = None


In [ ]:
import pandas as pd
from IPython.display import display

from model_secto import SectorModel
from research_secto import SectorResearch

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda x: f"{x:.6f}")


## 1. Configuration

Cette cellule ne lance pas encore le calcul. Elle permet de vérifier la composition du modèle avant exécution.


In [ ]:
model = SectorModel(
    universe=UNIVERSE,
    variables=MODEL_VARIABLES,
    excel_file=EXCEL_FILE,
    macro_file=MACRO_FILE,
)

display(model.summary())

print("Sectors:")
print(model.sectors)

print("\nAvailable variables:")
display(model.available_variables())

print("\nActive variables:")
display(model.variables())


## 2. Launch

Le calcul est exécuté une seule fois puis conservé dans le modèle.


In [ ]:
results = model.run()
research = SectorResearch(model)

print("Universe:", model.universe)
print("Latest pillar date:", max(df.index.max() for df in results["pillars"].values()))

if not results["history"].empty:
    print("Latest recommendation date:", results["history"]["date"].max())


## 3. Valeurs des variables

Affiche les variables actives du pilier et son score final par secteur.


In [ ]:
display(
    model.variable_table(
        pillar=PILLAR,
        date=DATE,
    )
)


## 4. Plot des piliers


In [ ]:
if PLOT_PILLARS:
    fig_pillars, ax_pillars = research.plot_pillars(
        date=DATE,
        pillars=PILLARS_PLOT,
    )
    display(fig_pillars)


## 5. Plot des variables


In [ ]:
if PLOT_VARIABLES:
    fig_variables, ax_variables = research.plot_variables(
        pillar=PILLAR,
        variables=BACKTEST_VARIABLES,
        date=DATE,
    )
    display(fig_variables)


## 6. Rang global


In [ ]:
if PLOT_GLOBAL_RANK:
    fig_rank, ax_rank = research.plot_global_rank(
        date=DATE,
    )
    display(fig_rank)


## 7. Backtest


In [ ]:
summary = None
backtests = None
fig_backtest = None

if RUN_BACKTEST:
    summary, backtests, fig_backtest, ax_backtest = research.backtest(
        pillar=PILLAR,
        variables=BACKTEST_VARIABLES,
        include_pillar=INCLUDE_PILLAR,
        top_n=TOP_N,
        start=START,
        end=END,
        with_plot=BACKTEST_PLOT,
        plot_series=PLOT_SERIES,
    )

    display(summary)

    if fig_backtest is not None:
        display(fig_backtest)


## 8. Détail mensuel


In [ ]:
if RUN_BACKTEST and backtests:
    variable = next(iter(backtests))
    print("Displayed variable:", variable)
    display(backtests[variable].tail(24))


## 9. Sauvegarde optionnelle


In [ ]:
if SAVE_BACKTEST and RUN_BACKTEST:
    research.save_backtest(
        summary=summary,
        backtests=backtests,
        pillar=PILLAR,
        output_dir=BACKTEST_OUTPUT_DIR,
    )
    print("Backtest saved in:", BACKTEST_OUTPUT_DIR)
